# 01b — Train: Stepwise Ablations (FT_PB and FT_FOLIO)

Trains two ablation conditions to isolate the contribution of each data
source in the proposed method (01a, FT_Combined).

## Conditions

**FT_PB** (`FlanT5_Stepwise_FT_PB`): fine-tunes on PlanBench fewshot stepwise
records only — no FOLIO data. Tests whether a small number of domain-specific
examples alone is sufficient for plan verification, and whether the model can
generalise to abstract Mystery domains without logical reasoning pre-training.

**FT_FOLIO** (`FlanT5_Stepwise_FT_FOLIO`): fine-tunes on FOLIO logical
entailment data only — no PlanBench examples. Tests whether logical reasoning
transfer alone (without any domain-specific adaptation) is sufficient.

Together, FT_PB and FT_FOLIO decompose the contribution of FT_Combined:
FT_FOLIO isolates the logical reasoning signal; FT_PB isolates the
domain-specific signal. Neither alone achieves the full performance of
FT_Combined (01a).

## Training data

| Condition | Data | Size |
|---|---|---|
| FT_PB | PlanBench fewshot stepwise only | 94 step records (25 fewshot_invalid plans) |
| FT_FOLIO | FOLIO train only | 677 examples (388 A, 289 B) |

## Output

Adapters saved to `paper2/adapters/FlanT5_Stepwise_FT_PB/` and
`paper2/adapters/FlanT5_Stepwise_FT_FOLIO/`.

## Prerequisites

Run `00_data_preparation.ipynb` first. `01a` does not need to be run before this.

## 1. Setup

DEBUG mode runs a quick 8-example / 2-epoch sanity check.
Set `DEBUG=False` for the actual training run.

Mount Google Drive, install dependencies, and log into HuggingFace.

In [ ]:
DEBUG   = False   # Set False for full training
DEBUG_N = 8
print(f'Mode: {"DEBUG" if DEBUG else "FULL TRAINING"}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes scikit-learn huggingface_hub

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via HF_TOKEN secret')
except Exception:
    from huggingface_hub import login
    login()

In [ ]:
import os, gc, json, random, math, warnings
import numpy as np
import torch
from collections import Counter
from dataclasses import dataclass as _dc

from datasets import load_dataset, Dataset as HFDataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,
    EarlyStoppingCallback, TrainerCallback, IntervalStrategy,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
DATA_DIR    = f'{ROOT}/data'
RESULTS_DIR = f'{ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

SW_FEWSHOT_PATH = os.path.join(DATA_DIR, 'planbench_stepwise_fewshot.jsonl')
MANIFEST_PATH   = os.path.join(DATA_DIR, 'split_manifest.json')

# ── Model & training config ───────────────────────────────────────────────────
MODEL_NAME     = 'google/flan-t5-xl'
MAX_SOURCE_LEN = 1024
MAX_TARGET_LEN = 4
PER_DEVICE_BS  = 2
GRAD_ACCUM     = 8       # effective batch = 2 × 8 = 16
MAX_EPOCHS     = 50      # ceiling for both conditions

DOMAINS = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']

use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
def cleanup(): gc.collect(); torch.cuda.empty_cache()

print(f'ROOT        : {ROOT}')
print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'BF16        : {use_bf16}')
print(f'MAX_EPOCHS  : {MAX_EPOCHS}')

## 2. Prompt Builder and Utilities

Both FOLIO and PlanBench records use the same entailment prompt template:

```
Task: Determine whether the conclusion is entailed or contradicted given the premises.
Premises:
- <premise 1>
  ...
Conclusion:
<conclusion>

Output format: Answer A if the conclusion is entailed, B if it is contradicted.
Answer:
```

This is identical to the prompt used in 01a. Using the same format for both
conditions ensures that differences in evaluation results are due to training
data composition, not prompt format differences.

In [ ]:
def build_folio_prompt(premises, conclusion):
    """Unified entailment prompt for both FOLIO and PlanBench records.
    For FOLIO: premises are FOL statements, conclusion is a proposition.
    For PlanBench: premises are domain rules + state facts, conclusion is an action.
    Output is always a single token: A (entailed/executable) or B (contradicted/invalid).
    """
    if isinstance(premises, (list, tuple)):
        prem_block = '\n'.join(
            '- ' + p.strip().rstrip('.') + '.'
            for p in premises if p.strip())
    else:
        prem_block = '- ' + str(premises).strip()
    return (
        'Task: Determine whether the conclusion is entailed or '
        'contradicted given the premises.\n'
        'Premises:\n' + prem_block + '\n\n'
        'Conclusion:\n' + conclusion.strip() + '\n\n'
        'Output format: Answer A if the conclusion is entailed, '
        'B if it is contradicted.\n'
        'Answer:'
    )

def normalize_pred(text):
    if not text: return 'B'
    c = text.strip()[0].upper()
    return c if c in {'A', 'B'} else 'B'

print('Prompt builder: OK')

## 3. Load PlanBench Fewshot Records

Loads the 94 stepwise action records from the 25 `fewshot_invalid` plans
built by `00_data_preparation.ipynb`. These are the same records used in
FT_Combined (01a).

Used by FT_PB only. FT_FOLIO does not use any PlanBench data.

In [ ]:
# Load all fewshot_invalid stepwise records
pb_fewshot_raw = [json.loads(l) for l in open(SW_FEWSHOT_PATH)]
print(f'Fewshot stepwise records loaded: {len(pb_fewshot_raw)}')
cnt = Counter(r['label'] for r in pb_fewshot_raw)
print(f'  A={cnt["A"]}  B={cnt["B"]}')
print(f'  Class ratio A:B = {cnt["A"]/max(1,cnt["B"]):.1f}:1  '
      f'(handled by weighted loss)')
for domain in DOMAINS:
    dom_recs = [r for r in pb_fewshot_raw if r['domain'] == domain]
    dom_cnt  = Counter(r['label'] for r in dom_recs)
    n_plans  = len(set(r['plan_id'] for r in dom_recs))
    print(f'  {domain:15s}: {len(dom_recs):3d} steps from {n_plans} plans  '
          f'(A={dom_cnt["A"]} B={dom_cnt["B"]})')


## 4. Load FOLIO Dataset

FOLIO is a human-annotated natural language reasoning dataset grounded in
first-order logic. Each example has premises, a conclusion, and a label:
True (entailed → A), False (contradicted → B), or Uncertain (dropped).
After dropping Uncertain: 677 training examples (388 A, 289 B),
134 validation examples (72 A, 62 B).

Used by FT_FOLIO only. FT_PB does not use any FOLIO data.

In [ ]:
LABEL_MAP = {'True': 'A', 'False': 'B'}
ALT       = {'true': 'True', 'false': 'False', True: 'True', False: 'False'}

def norm_label(raw):
    s = ALT.get(raw, str(raw).strip())
    return LABEL_MAP.get(s, None)

raw_folio = load_dataset('yale-nlp/FOLIO')
print('FOLIO train:', Counter(raw_folio['train']['label']))
print('FOLIO val  :', Counter(raw_folio['validation']['label']))

def process_folio_split(split):
    out = []
    for ex in raw_folio[split]:
        lbl = norm_label(ex['label'])
        if lbl is None: continue
        out.append({
            'user_text'   : build_folio_prompt(ex['premises'], ex['conclusion']),
            'label_letter': lbl
        })
    return out

folio_train = HFDataset.from_list(process_folio_split('train'))
folio_val   = HFDataset.from_list(process_folio_split('validation'))
print(f'\nFOLIO train: {len(folio_train)}  val: {len(folio_val)}')

## 5. Tokeniser

Flan-T5-XL encoder-decoder tokeniser. Input prompts are tokenised for
the encoder; target labels (`A` or `B`) for the decoder.

- `MAX_SOURCE_LEN = 1024`: sufficient for all stepwise prompts
- `MAX_TARGET_LEN = 4`: target is always a single token

Shared between both FT_PB and FT_FOLIO training runs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['user_text'],
        max_length=MAX_SOURCE_LEN, truncation=True, padding=False)
    labels_enc = tokenizer(
        text_target=batch['label_letter'],
        max_length=MAX_TARGET_LEN, truncation=True, padding=False)
    pad_id = tokenizer.pad_token_id
    model_inputs['labels'] = [
        [(t if t != pad_id else -100) for t in seq]
        for seq in labels_enc['input_ids']
    ]
    return model_inputs

A_ID = tokenizer.encode('A', add_special_tokens=False)[0]
B_ID = tokenizer.encode('B', add_special_tokens=False)[0]
print(f'Tokenizer loaded. Token IDs: A={A_ID}  B={B_ID}')

## 6. Model Builder and Shared Training Components

### LoRA configuration
Same as 01a: r=8, alpha=16, dropout=0.05, target modules q/k/v/o.
Trainable parameters: ~9.4M / 2.86B (0.33%).

### Weighted cross-entropy loss
`w_c = N / (2 × N_c)` computed per condition from its own training label
distribution. FT_PB has more extreme imbalance (94 steps, mostly A) so its
weights differ from FT_FOLIO (677 FOLIO examples, more balanced).

### Gradient checkpointing
Enabled for both conditions to fit Flan-T5-XL on a single GPU.

In [ ]:
class WeightedSeq2SeqTrainer(Seq2SeqTrainer):
    """
    Weighted cross-entropy loss trainer.
    The stepwise fewshot training data is heavily class-imbalanced:
    most steps are valid (label A) since only the single failing step
    in an invalid plan gets label B. Without correction, the model
    collapses to always predicting A.

    We apply inverse-frequency weighting: a weight vector of shape
    (vocab_size,) is set to 1.0 everywhere, then w[A_ID] and w[B_ID]
    are set to their respective inverse-frequency weights. All other
    positions are masked to -100 in labels so they never contribute.
    """
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            w = torch.ones(self.model.config.vocab_size,
                           device=next(self.model.parameters()).device)
            w[A_ID] = class_weights[0]
            w[B_ID] = class_weights[1]
            self.loss_fct = torch.nn.CrossEntropyLoss(weight=w, ignore_index=-100)
        else:
            self.loss_fct = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.loss_fct is None:
            return super().compute_loss(model, inputs, return_outputs, **kwargs)
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = self.loss_fct(
            logits.view(-1, logits.size(-1)),
            labels.view(-1).to(logits.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    """F1-macro on whatever val set is passed — used for both conditions."""
    pred_ids, label_ids = eval_pred
    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids  = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    preds = [normalize_pred(t) for t in pred_texts]
    golds = [normalize_pred(t) for t in gold_texts]
    acc  = sum(p == g for p, g in zip(preds, golds)) / max(1, len(golds))
    f1   = f1_score(
        [1 if g == 'A' else 0 for g in golds],
        [1 if p == 'A' else 0 for p in preds],
        average='macro', zero_division=0)
    return {'accuracy': acc, 'f1_macro': f1}

def build_model():
    """Load Flan-T5-XL and apply LoRA. Called fresh for each training run."""
    cleanup()
    base = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.bfloat16 if use_bf16 else torch.float32,
        device_map='auto',
    )
    base.gradient_checkpointing_enable()
    if hasattr(base, 'enable_input_require_grads'):
        base.enable_input_require_grads()
    model = get_peft_model(base, LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=['q', 'k', 'v', 'o'],
    ))
    if hasattr(model, 'generation_config'):
        model.generation_config.max_length = MAX_TARGET_LEN + 1
    return model

def get_class_weights(records):
    """Compute inverse-frequency weights from a list of records with 'label' field."""
    n_A = sum(1 for r in records if r['label'] == 'A')
    n_B = sum(1 for r in records if r['label'] == 'B')
    n_total = n_A + n_B
    w_A = n_total / (2 * n_A)
    w_B = n_total / (2 * n_B)
    return w_A, w_B

def make_training_args(output_dir, n_epochs, n_train):
    """
    Build Seq2SeqTrainingArguments with dynamic eval_steps and warmup_steps
    scaled to the dataset size. Prevents eval firing too rarely on small datasets
    or warmup exceeding total training steps.
    """
    steps_per_epoch = max(1, math.ceil(n_train / (PER_DEVICE_BS * GRAD_ACCUM)))
    total_steps     = steps_per_epoch * (2 if DEBUG else n_epochs)
    eval_steps      = max(1, min(200, total_steps // 5))  # eval ~5 times per training
    warmup_steps    = max(1, min(65, total_steps // 10))
    print(f'  steps/epoch={steps_per_epoch}  total={total_steps}  '
          f'eval_steps={eval_steps}  warmup={warmup_steps}')
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=PER_DEVICE_BS,
        per_device_eval_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=2 if DEBUG else n_epochs,
        learning_rate=2e-4,
        warmup_steps=warmup_steps,
        lr_scheduler_type='cosine',
        weight_decay=0.0,
        eval_strategy=IntervalStrategy.STEPS,
        eval_steps=5 if DEBUG else eval_steps,
        save_steps=5 if DEBUG else eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1_macro',
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,
        logging_steps=1 if DEBUG else 50,
        report_to='none',
        fp16=False, bf16=bool(use_bf16),
        dataloader_num_workers=2, seed=SEED,
    )

print('Shared training components: OK')

## 7. FT_PB Training (FlanT5_Stepwise_FT_PB)

Trains on the 94 PlanBench fewshot stepwise records only. No FOLIO data.

**Fixed 30 epochs, no early stopping.**
With only 94 step records, any held-out validation split (~20 records) is
too small for stable F1-macro computation. Fixed epochs are more robust.
30 epochs = ~180 gradient updates (effective batch size 16).

**What this tests:** whether domain-specific stepwise examples alone are
sufficient for plan verification, and whether the model generalises to
abstract Mystery domains without logical reasoning pre-training.
Expected result: good performance on concrete domains (Blocksworld, Logistics)
but poor performance on Mystery domains where abstract vocabulary requires
genuine logical inference rather than vocabulary matching.

Adapter saved to `paper2/adapters/FlanT5_Stepwise_FT_PB/`.

In [ ]:
def records_to_ds(records):
    return HFDataset.from_list([
        {'user_text': r['prompt'], 'label_letter': r['label']}
        for r in records
    ])

FT_PB_EPOCHS = 30   # fixed epoch count

pb_train_ds = records_to_ds(pb_fewshot_raw)

if DEBUG:
    pb_train_ds = pb_train_ds.select(range(min(DEBUG_N, len(pb_train_ds))))
    print(f'[DEBUG] train={len(pb_train_ds)}')

print(f'FT_PB train: {len(pb_train_ds)} steps  '
      f'({len(set(r["plan_id"] for r in pb_fewshot_raw))} plans, '
      f'all 5 fewshot_invalid per domain)')

w_A_pb, w_B_pb = get_class_weights(pb_fewshot_raw)

# Tokenize
tok_pb = pb_train_ds.map(tokenize_batch, batched=True,
                         remove_columns=pb_train_ds.column_names)

PB_ADAPTER_DIR = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_PB', 'lora_adapter')
PB_TOK_DIR     = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_PB', 'tokenizer')
PB_CKPT_DIR    = os.path.join(ROOT, 'checkpoints', 'FlanT5_Stepwise_FT_PB')
os.makedirs(os.path.dirname(PB_ADAPTER_DIR), exist_ok=True)
os.makedirs(PB_CKPT_DIR, exist_ok=True)

steps_per_epoch_pb = max(1, math.ceil(len(tok_pb) / (PER_DEVICE_BS * GRAD_ACCUM)))
total_steps_pb     = steps_per_epoch_pb * (2 if DEBUG else FT_PB_EPOCHS)
warmup_pb          = max(1, min(65, total_steps_pb // 10))
print(f'steps/epoch={steps_per_epoch_pb}  '
      f'total_steps={total_steps_pb}  warmup={warmup_pb}')

model_pb = build_model()
model_pb.print_trainable_parameters()

args_pb = Seq2SeqTrainingArguments(
    output_dir=PB_CKPT_DIR,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=2 if DEBUG else FT_PB_EPOCHS,
    learning_rate=2e-4,
    warmup_steps=warmup_pb,
    lr_scheduler_type='cosine',
    weight_decay=0.0,
    eval_strategy='no',
    save_strategy='no',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=1 if DEBUG else 20,
    report_to='none',
    fp16=False, bf16=bool(use_bf16),
    dataloader_num_workers=2, seed=SEED,
)

trainer_pb = WeightedSeq2SeqTrainer(
    model=model_pb, args=args_pb,
    train_dataset=tok_pb,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model_pb, padding=True),
    class_weights=(w_A_pb, w_B_pb),
)

trainer_pb.train()

trainer_pb.model.save_pretrained(PB_ADAPTER_DIR)
tokenizer.save_pretrained(PB_TOK_DIR)
print(f'\nAdapter   -> {PB_ADAPTER_DIR}')
print(f'Tokenizer -> {PB_TOK_DIR}')
print(f'Epochs    = {FT_PB_EPOCHS}')

del model_pb, trainer_pb
cleanup()


## 8. FT_FOLIO Training (FlanT5_Stepwise_FT_FOLIO)

Trains on FOLIO logical entailment data only. No PlanBench examples.

**Early stopping, patience=3 on FOLIO validation F1-macro.**
677 training examples with 134 validation examples — sufficient for stable
early stopping. Max 50 epochs.

**What this tests:** whether logical reasoning transfer from FOLIO alone
is sufficient for plan verification, without any planning domain vocabulary
adaptation. The model learns to derive conclusions from premises but has
never seen blocksworld block names, logistics locations, or mystery domain
actions.

Expected result: better performance on abstract Mystery domains (where
vocabulary is already abstract, so FOLIO reasoning transfers directly) but
weaker performance on concrete domains (where the model must bridge from
abstract FOL premises to physical action descriptions).

Adapter saved to `paper2/adapters/FlanT5_Stepwise_FT_FOLIO/`.

In [ ]:
# Class weights from FOLIO train
all_folio_labels = folio_train['label_letter']
n_A_f = all_folio_labels.count('A')
n_B_f = all_folio_labels.count('B')
n_total_f = n_A_f + n_B_f
w_A_f = n_total_f / (2 * n_A_f)
w_B_f = n_total_f / (2 * n_B_f)
print(f'FOLIO class balance : A={n_A_f} ({n_A_f/n_total_f*100:.1f}%)  '
      f'B={n_B_f} ({n_B_f/n_total_f*100:.1f}%)')
print(f'FOLIO class weights : A={w_A_f:.3f}  B={w_B_f:.3f}')

folio_train_use = folio_train
folio_val_use   = folio_val
if DEBUG:
    folio_train_use = folio_train.select(range(min(DEBUG_N, len(folio_train))))
    folio_val_use   = folio_val.select(range(min(DEBUG_N, len(folio_val))))
    print(f'[DEBUG] train={len(folio_train_use)} val={len(folio_val_use)}')

ds_folio = DatasetDict({'train': folio_train_use, 'validation': folio_val_use})
tok_folio = ds_folio.map(tokenize_batch, batched=True,
                          remove_columns=ds_folio['train'].column_names)
print(f'\nFOLIO train: {len(tok_folio["train"])}  val: {len(tok_folio["validation"])}')

FOLIO_ADAPTER_DIR = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_FOLIO', 'lora_adapter')
FOLIO_TOK_DIR     = os.path.join(ROOT, 'adapters', 'FlanT5_Stepwise_FT_FOLIO', 'tokenizer')
FOLIO_CKPT_DIR    = os.path.join(ROOT, 'checkpoints', 'FlanT5_Stepwise_FT_FOLIO')
os.makedirs(os.path.dirname(FOLIO_ADAPTER_DIR), exist_ok=True)
os.makedirs(FOLIO_CKPT_DIR, exist_ok=True)

print('\n--- FlanT5_Stepwise_FT_FOLIO Training ---')
model_folio = build_model()
model_folio.print_trainable_parameters()

args_folio = make_training_args(FOLIO_CKPT_DIR, MAX_EPOCHS, len(folio_train_use))

trainer_folio = WeightedSeq2SeqTrainer(
    model=model_folio, args=args_folio,
    train_dataset=tok_folio['train'],
    eval_dataset=tok_folio['validation'],
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model_folio, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    class_weights=(w_A_f, w_B_f),
)

trainer_folio.train()
best_folio = trainer_folio.state.best_metric
print(f'\nBest FOLIO val F1-macro: {best_folio:.4f}'
      if best_folio else 'Best: N/A')

trainer_folio.model.save_pretrained(FOLIO_ADAPTER_DIR)
tokenizer.save_pretrained(FOLIO_TOK_DIR)
print(f'Adapter   → {FOLIO_ADAPTER_DIR}')
print(f'Tokenizer → {FOLIO_TOK_DIR}')

del model_folio, trainer_folio
cleanup()

## 9. FOLIO Validation Report — FT_FOLIO

Runs inference on the full FOLIO validation set (134 examples) with the
best FT_FOLIO checkpoint and prints a classification report.

Confirms that the best checkpoint has genuinely learned logical entailment
on the held-out FOLIO validation set.

In [ ]:
from peft import PeftModel

def folio_val_report(adapter_dir, tok_dir, label):
    if not os.path.isdir(adapter_dir):
        print(f'[SKIP] {label}: adapter not found at {adapter_dir}')
        return
    cleanup()
    tok  = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)
    base = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.bfloat16 if use_bf16 else torch.float32,
        device_map='auto',
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    preds_out, golds_out = [], []
    for ex in folio_val:
        inp = tok(ex['user_text'], return_tensors='pt',
                  max_length=MAX_SOURCE_LEN, truncation=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=MAX_TARGET_LEN)
        pred = tok.decode(out[0], skip_special_tokens=True)
        preds_out.append(normalize_pred(pred))
        golds_out.append(ex['label_letter'])
    lmap = {'A': 'Entailed', 'B': 'Contradicted'}
    print(f'\n=== FOLIO VAL REPORT — {label} ===')
    print(classification_report(
        [lmap.get(g, '?') for g in golds_out],
        [lmap.get(p, '?') for p in preds_out],
        labels=['Entailed', 'Contradicted'], zero_division=0))
    del model; cleanup()

folio_val_report(FOLIO_ADAPTER_DIR, FOLIO_TOK_DIR, 'FlanT5_Stepwise_FT_FOLIO')

## 10. Summary

Training complete for both ablation conditions. Results on PlanBench are
reported in `02_eval_flant5.ipynb` after running inference on the test set.

Key expected findings:
- FT_PB: high plan F1 on Blocksworld and Logistics, near-chance on Mystery
- FT_FOLIO: moderate plan F1 across all domains including Mystery,
  demonstrating that logical reasoning transfers to abstract vocabulary
- Neither condition matches FT_Combined, confirming both data sources
  are necessary

In [ ]:
print('=== 01b TRAINING COMPLETE ===')
print()
print('FlanT5_Stepwise_FT_PB')
print(f'  Training data : all 5 fewshot_invalid plans/domain (~120 step records)')
print(f'  Fixed epochs  : {FT_PB_EPOCHS}')
print(f'  Adapter       : {PB_ADAPTER_DIR}')
print()
print('FlanT5_Stepwise_FT_FOLIO')
print(f'  Training data : FOLIO only (~677 examples)')
print(f'  Adapter       : {FOLIO_ADAPTER_DIR}')
print()
print('Next step: run 01c_train_direct_ft.ipynb')
